# Custom Simulation Design Tutorial

This tutorial explains how to design custom simulations in chance_c by creating your own agents and engines. We'll walk through the architecture and show you how to extend the simulation framework with custom behaviors.

## Learning Objectives

By the end of this tutorial, you will understand:
1. The architecture of chance_c simulations
2. How to create custom agents (both urban and institutional)
3. How to create custom engines for specific behaviors
4. How to integrate custom components into a simulation
5. Best practices for extending the simulation framework

## Prerequisites

- Basic understanding of Python and object-oriented programming
- Familiarity with the chance_c package structure
- Understanding of agent-based modeling concepts


In [ ]:
# Import required packages
import logging
import random
import numpy as np
import pandas as pd
from typing import Dict, List, Optional, Union

# Import chance_c components
from chance_c import (
    Model, SimulationConfig, 
    ICOMSimulator, HouseholdAgent, 
    AllHouseholdAgents, CountyZoningManager, RealEstate,
    NewAgentCreation, HousingMarket, BuildingDevelopment
)

# Import pynsim base classes for custom components
from pynsim import Engine, Institution
from pynsim.components.component import Component

# Set up logging
logging.basicConfig(level=logging.INFO)


## Part 1: Understanding the Simulation Architecture

The chance_c simulation framework is built on pynsim and consists of several key components:

### Core Components

1. **Simulator (ICOMSimulator)**: The main simulation engine that orchestrates all components
2. **Network**: Contains nodes (BlockGroups) and manages the spatial structure
3. **Agents**: Individual entities that make decisions (e.g., HouseholdAgent)
4. **Institutions**: Organizations that manage groups of agents or make collective decisions
5. **Engines**: Behavioral modules that execute specific simulation logic each timestep

### Architecture Flow

```
Simulator
├── Network (spatial structure)
│   ├── Nodes (BlockGroups)
│   └── Components (Agents)
├── Institutions (agent collections/managers)
│   ├── AllHouseholdAgents
│   ├── CountyZoningManager
│   └── RealEstate
└── Engines (behavioral logic)
    ├── NewAgentCreation
    ├── HousingMarket
    ├── BuildingDevelopment
    └── [Your Custom Engines]
```

The simulation runs by executing all engines in sequence for each timestep, allowing agents to make decisions and update the system state.


## Understanding Model.run_simulation() Key Steps

The `Model.run_simulation()` method performs several key steps that we can use as a template for our custom simulation design:

### 🔍 Key Steps in Model.run_simulation():

1. **Create ICOMSimulator instance**
2. **Set timestep information** (start_year, n_years)
3. **Set landscape** (load geographic and demographic data)
4. **Create institutions** (AllHouseholdAgents, CountyZoningManager, RealEstate)
5. **Convert initial population to agents**
6. **Initialize housing units and vacancy**
7. **Add engines to simulator:**
   - NewAgentCreation (population growth)
   - ExistingAgentReloSampler (who wants to move)
   - NewAgentLocation (where new agents move)
   - ExistingAgentLocation (where existing agents move)
   - HousingMarket (match buyers with sellers)
   - BuildingDevelopment (construct new housing)
   - HousingPricing (update prices)
   - FloodHazard (environmental impacts)
   - Zoning (regulatory decisions)
   - LandscapeStatistics (track metrics)
8. **Start simulation** (simulator.start())

### 💡 Customization Opportunities

To create a custom simulation, we can:

- **Replace any of these engines** with custom versions
- **Add new engines** for additional behaviors
- **Create new agent types** with different decision-making
- **Add new institutions** for collective decision-making

This modular design makes it easy to extend chance_c with your own custom components while maintaining compatibility with the existing framework.


## Part 2: Creating Custom Agents

Agents are the individual entities that make decisions in the simulation. chance_c has two main types of agents:

1. **Urban Agents**: Individual decision-makers (like HouseholdAgent)
2. **Institutional Agents**: Organizations that manage resources or make collective decisions

Let's create custom agents of both types.


In [ ]:
# Example 1: Custom Urban Agent - Business Agent
class BusinessAgent(Component):
    """
    A custom agent representing a business that makes location decisions
    based on labor availability, transportation access, and zoning.
    """
    
    def __init__(
        self, 
        name: str, 
        location: str = None,
        business_type: str = 'retail',
        employees: int = 10,
        revenue: float = 500000,
        location_preferences: Dict[str, float] = None,
        **kwargs
    ) -> None:
        """Initialize the BusinessAgent.
        
        Args:
            name: Unique identifier for the business
            location: Current location (block group ID)
            business_type: Type of business (retail, office, industrial, etc.)
            employees: Number of employees
            revenue: Annual revenue
            location_preferences: Dictionary of location preference weights
        """
        super(BusinessAgent, self).__init__(name, **kwargs)
        self.location = location
        self.business_type = business_type
        self.employees = employees
        self.revenue = revenue
        self.location_preferences = location_preferences or {
            'labor_availability': 0.4,
            'transportation_access': 0.3,
            'zoning_compatibility': 0.2,
            'cost': 0.1
        }
        self.location_utilities = {}  # Will store calculated utilities for locations
    
    # Properties that can be tracked over time
    _properties = {
        'location': None,
        'employees': 0,
        'revenue': 0.0,
        'location_utilities': {}
    }
    
    def setup(self, timestep: int) -> None:
        """Set up the business agent for a given timestep."""
        self.location_utilities = {}  # Reset utilities each timestep
    
    def calculate_location_utility(self, block_group_id: str, network) -> float:
        """
        Calculate utility for a potential business location.
        
        Args:
            block_group_id: ID of the block group to evaluate
            network: The simulation network containing data
            
        Returns:
            float: Calculated utility score for the location
        """
        # Get block group data (this would access real data in a full simulation)
        # For demonstration, we'll use placeholder calculations
        
        # Labor availability (based on population density)
        labor_score = random.uniform(0.3, 1.0)  # Placeholder
        
        # Transportation access (distance to major roads/transit)
        transport_score = random.uniform(0.2, 1.0)  # Placeholder
        
        # Zoning compatibility (whether business type is allowed)
        zoning_score = 1.0 if random.random() > 0.2 else 0.0  # 80% compatible
        
        # Cost (inverse of land/rent prices)
        cost_score = random.uniform(0.4, 0.9)  # Placeholder
        
        # Calculate weighted utility
        utility = (
            self.location_preferences['labor_availability'] * labor_score +
            self.location_preferences['transportation_access'] * transport_score +
            self.location_preferences['zoning_compatibility'] * zoning_score +
            self.location_preferences['cost'] * cost_score
        )
        
        self.location_utilities[block_group_id] = utility
        return utility
    
    def decide_relocation(self, threshold: float = 0.7) -> bool:
        """
        Decide whether to relocate based on current location utility.
        
        Args:
            threshold: Utility threshold below which business considers relocating
            
        Returns:
            bool: True if business wants to relocate
        """
        if self.location and self.location in self.location_utilities:
            current_utility = self.location_utilities[self.location]
            return current_utility < threshold
        return False

# Test the custom BusinessAgent
print("Custom BusinessAgent Example:")
business = BusinessAgent(
    name="tech_startup_001",
    business_type="office",
    employees=25,
    revenue=750000,
    location_preferences={
        'labor_availability': 0.5,
        'transportation_access': 0.3,
        'zoning_compatibility': 0.1,
        'cost': 0.1
    }
)

# Demonstrate utility calculation
utility = business.calculate_location_utility("block_group_123", None)
print(f"  Business: {business.name}")
print(f"  Type: {business.business_type}")
print(f"  Employees: {business.employees}")
print(f"  Calculated utility for location: {utility:.3f}")
print(f"  Would relocate: {business.decide_relocation()}")
print("  BusinessAgent created successfully!")


In [ ]:
# Example 2: Custom Institutional Agent
class TransmissionPlanningAgency(Institution):
    """
    A custom institutional agent that manages electricity transmission planning.
    """
    
    def __init__(self, name: str, **kwargs) -> None:
        """Initialize the Transmission Planning Agency.
        
        Args:
            name: Name identifier for the institution
        """
        super(TransmissionPlanningAgency, self).__init__(name, **kwargs)
        
        # Transmission planning parameters
        self.capacity_threshold = 0.85  # 85% capacity threshold for upgrades
        self.reliability_standards = {
            'n1_contingency': True,  # N-1 reliability standard
            'voltage_limits': {'min': 0.95, 'max': 1.05},  # per unit
            'thermal_limits': 0.9  # 90% of thermal capacity
        }
        self.transmission_corridors = set()  # Protected transmission corridors
        self.grid_investments = {}  # Track transmission investments
    
    def setup(self, timestep: int) -> None:
        """Set up the Transmission Planning Agency for a given timestep."""
        pass
    
    def assess_transmission_needs(self, block_group_id: str, load_growth: float) -> Dict[str, Union[bool, str]]:
        """
        Assess transmission needs for load growth in a block group.
        
        Args:
            block_group_id: ID of block group for load growth
            load_growth: Expected load growth in MW
            
        Returns:
            dict: Assessment results with approval status and recommendations
        """
        assessment = {
            'approved': True,
            'reasons': [],
            'conditions': [],
            'investment_needed': 0.0
        }
        
        # Simulate transmission system checks (in real simulation, would use actual data)
        current_capacity = random.uniform(50, 200)  # MW
        available_capacity = current_capacity * random.uniform(0.6, 0.95)
        voltage_stability = random.uniform(0.92, 1.08)
        right_of_way_available = random.random() < 0.7  # 70% chance of available ROW
        
        # Check capacity adequacy
        if available_capacity < load_growth:
            assessment['approved'] = False
            assessment['reasons'].append(f"Insufficient capacity: {available_capacity:.1f} MW available, {load_growth:.1f} MW needed")
            assessment['investment_needed'] = load_growth - available_capacity
        
        # Check voltage stability
        if voltage_stability < self.reliability_standards['voltage_limits']['min']:
            assessment['approved'] = False
            assessment['reasons'].append(f"Voltage stability issue: {voltage_stability:.3f} pu")
        
        # Check right of way availability
        if not right_of_way_available:
            assessment['approved'] = False
            assessment['reasons'].append("No available transmission right of way")
        
        # Add conditions if approved
        if assessment['approved']:
            if available_capacity < current_capacity * self.capacity_threshold:
                assessment['conditions'].append("Transmission upgrade recommended")
            if load_growth > 50:  # MW threshold for conditions
                assessment['conditions'].append("Substation expansion required")
        
        return assessment
    
    def designate_transmission_corridor(self, block_group_id: str, reason: str) -> None:
        """
        Designate a block group as a transmission corridor.
        
        Args:
            block_group_id: ID of block group to designate as corridor
            reason: Reason for corridor designation
        """
        self.transmission_corridors.add(block_group_id)
        print(f"⚡ TPA designated {block_group_id} as transmission corridor: {reason}")
    
    def approve_investment(self, block_group_id: str, project_type: str, cost: float) -> None:
        """
        Approve transmission investment project.
        
        Args:
            block_group_id: Location of investment
            project_type: Type of transmission project
            cost: Project cost in millions
        """
        if block_group_id not in self.grid_investments:
            self.grid_investments[block_group_id] = []
        
        investment = {
            'type': project_type,
            'cost': cost,
            'timestamp': 'current_timestep'  # Would use actual timestep
        }
        self.grid_investments[block_group_id].append(investment)
        print(f"  TPA approved investment: {project_type} at {block_group_id}, cost: ${cost:,.1f}M")

# Test the custom Transmission Planning Agency institution
print("⚡ Custom Transmission Planning Agency Example:")
tpa = TransmissionPlanningAgency(name="county_tpa")

# Test transmission assessment
assessment = tpa.assess_transmission_needs("block_group_456", 75.0)
print(f"  Transmission Assessment:")
print(f"    Approved: {assessment['approved']}")
if assessment['reasons']:
    print(f"    Reasons: {', '.join(assessment['reasons'])}")
if assessment['conditions']:
    print(f"    Conditions: {', '.join(assessment['conditions'])}")
if assessment['investment_needed'] > 0:
    print(f"    Investment needed: {assessment['investment_needed']:.1f} MW")

# Test corridor designation
tpa.designate_transmission_corridor("corridor_001", "Major transmission route")

# Test investment approval
tpa.approve_investment("substation_002", "Substation upgrade", 25.5)

print(f"  Transmission corridors: {len(tpa.transmission_corridors)}")
print(f"  Investments approved: {len(tpa.grid_investments)}")
print("  Transmission Planning Agency created successfully!")


## Part 3: Creating Custom Engines

Engines are the behavioral modules that execute specific simulation logic each timestep. They operate on agents and update the simulation state. Let's create custom engines that work with our new agents.


In [ ]:
# Example 1: Custom Engine - Business Location Engine
class BusinessLocationEngine(Engine):
    """
    An engine that handles business location decisions and relocation.
    This engine processes all business agents and helps them find optimal locations.
    """
    
    def __init__(
        self, 
        target, 
        relocation_probability: float = 0.15,
        sample_size: int = 5,
        **kwargs
    ) -> None:
        """Initialize the BusinessLocationEngine.
        
        Args:
            target: The simulation network target
            relocation_probability: Probability that a business considers relocating
            sample_size: Number of locations to sample for each business
        """
        super(BusinessLocationEngine, self).__init__(target, **kwargs)
        self.relocation_probability = relocation_probability
        self.sample_size = sample_size
    
    def run(self) -> None:
        """Execute the business location engine logic."""
        logging.info(f"Running BusinessLocationEngine, year {self.timestep.year}")
        
        # Get all business agents (would need to be stored in a business institution)
        business_agents = self._get_business_agents()
        
        relocating_businesses = []
        
        # Step 1: Determine which businesses want to relocate
        for business in business_agents:
            # Random chance of considering relocation
            if random.random() < self.relocation_probability:
                # Calculate utility for current location
                if business.location:
                    current_utility = business.calculate_location_utility(business.location, self.target)
                    if business.decide_relocation():
                        relocating_businesses.append(business)
                        print(f"  {business.name} considering relocation (current utility: {current_utility:.3f})")
        
        # Step 2: Find new locations for relocating businesses
        for business in relocating_businesses:
            best_location = self._find_best_location(business)
            if best_location and best_location != business.location:
                self._relocate_business(business, best_location)
        
        # Step 3: Update business statistics
        self._update_business_statistics(business_agents)
    
    def _get_business_agents(self) -> List[BusinessAgent]:
        """Get all business agents from the simulation."""
        # In a real simulation, businesses would be stored in an institution
        # For demonstration, we'll create some sample businesses
        if not hasattr(self.target, 'business_agents'):
            self.target.business_agents = [
                BusinessAgent(f"business_{i}", business_type=random.choice(['retail', 'office', 'industrial']))
                for i in range(3)
            ]
        return self.target.business_agents
    
    def _find_best_location(self, business: BusinessAgent) -> Optional[str]:
        """
        Find the best location for a business by sampling available locations.
        
        Args:
            business: The business agent looking for a location
            
        Returns:
            str: ID of the best location, or None if no suitable location found
        """
        # Sample available locations (in real simulation, would use actual block groups)
        available_locations = [f"block_group_{i}" for i in range(10)]
        sampled_locations = random.sample(available_locations, min(self.sample_size, len(available_locations)))
        
        best_location = None
        best_utility = -1
        
        for location in sampled_locations:
            utility = business.calculate_location_utility(location, self.target)
            if utility > best_utility:
                best_utility = utility
                best_location = location
        
        return best_location if best_utility > 0.5 else None  # Minimum utility threshold
    
    def _relocate_business(self, business: BusinessAgent, new_location: str) -> None:
        """
        Relocate a business to a new location.
        
        Args:
            business: The business agent to relocate
            new_location: ID of the new location
        """
        old_location = business.location
        business.location = new_location
        
        # Update location tracking (would update block group data in real simulation)
        print(f"  {business.name} relocated from {old_location} to {new_location}")
        
        # Record relocation in business history
        if not hasattr(business, 'relocation_history'):
            business.relocation_history = []
        business.relocation_history.append({
            'from': old_location,
            'to': new_location,
            'year': self.timestep.year
        })
    
    def _update_business_statistics(self, business_agents: List[BusinessAgent]) -> None:
        """Update aggregate business statistics."""
        total_businesses = len(business_agents)
        business_types = {}
        
        for business in business_agents:
            business_types[business.business_type] = business_types.get(business.business_type, 0) + 1
        
        # Store statistics (would be saved to network history in real simulation)
        if not hasattr(self.target, 'business_statistics'):
            self.target.business_statistics = []
        
        stats = {
            'year': self.timestep.year,
            'total_businesses': total_businesses,
            'business_types': business_types
        }
        self.target.business_statistics.append(stats)

# Test the BusinessLocationEngine
print("  Custom BusinessLocationEngine Example:")

# Create a mock simulation target with timestep
class MockTarget:
    def __init__(self):
        self.business_agents = []
        self.business_statistics = []

class MockTimestep:
    def __init__(self, year):
        self.year = year

# Create and test the engine
target = MockTarget()
engine = BusinessLocationEngine(target, relocation_probability=0.5)
engine.timestep = MockTimestep(2023)

# Run the engine
engine.run()

print(f"  Business statistics: {len(target.business_statistics)} years recorded")
if target.business_statistics:
    latest_stats = target.business_statistics[-1]
    print(f"  Total businesses: {latest_stats['total_businesses']}")
    print(f"  Business types: {latest_stats['business_types']}")

print("  BusinessLocationEngine created and tested successfully!")


In [ ]:
# Example 2: Custom Engine - Electricity Transmission Planning Engine
class TransmissionPlanningEngine(Engine):
    """
    An engine that manages electricity transmission planning and infrastructure development.
    This engine coordinates transmission projects, assesses grid reliability, manages capacity,
    and handles regulatory compliance for transmission infrastructure.
    """
    
    def __init__(
        self, 
        target, 
        project_review_probability: float = 0.4,
        planning_update_interval: int = 3,
        capacity_assessment_frequency: int = 2,
        **kwargs
    ) -> None:
        """Initialize the TransmissionPlanningEngine.
        
        Args:
            target: The transmission planning authority that manages grid infrastructure
            project_review_probability: Probability of reviewing transmission projects per timestep
            planning_update_interval: Years between major planning updates
            capacity_assessment_frequency: Years between capacity assessments
        """
        super(TransmissionPlanningEngine, self).__init__(target, **kwargs)
        self.project_review_probability = project_review_probability
        self.planning_update_interval = planning_update_interval
        self.capacity_assessment_frequency = capacity_assessment_frequency
        self.last_planning_update = 0
        self.last_capacity_assessment = 0
    
    def run(self) -> None:
        """Execute the transmission planning engine logic."""
        logging.info(f"Running TransmissionPlanningEngine, year {self.timestep.year}")
        
        # Step 1: Review transmission infrastructure projects
        self._review_transmission_projects()
        
        # Step 2: Assess grid capacity and reliability
        if self.timestep.year - self.last_capacity_assessment >= self.capacity_assessment_frequency:
            self._assess_grid_capacity()
            self.last_capacity_assessment = self.timestep.year
        
        # Step 3: Update transmission planning periodically
        if self.timestep.year - self.last_planning_update >= self.planning_update_interval:
            self._update_transmission_planning()
            self.last_planning_update = self.timestep.year
        
        # Step 4: Coordinate with regional transmission organizations
        self._coordinate_regional_planning()
        
        # Step 5: Manage transmission line maintenance and upgrades
        self._manage_maintenance_schedule()
        
        # Step 6: Assess renewable energy integration needs
        self._assess_renewable_integration()
    
    def _review_transmission_projects(self) -> None:
        """Review electricity transmission infrastructure projects."""
        # Simulate transmission project reviews
        num_projects = random.randint(2, 6)
        
        for i in range(num_projects):
            if random.random() < self.project_review_probability:
                project_id = f"transmission_project_{i}"
                project_type = random.choice(['high_voltage_line', 'substation', 'transformer', 'smart_grid', 'battery_storage'])
                
                # Assess transmission project feasibility
                assessment = self._assess_transmission_project(project_id, project_type)
                
                if assessment['approved']:
                    print(f"Transmission project approved: {project_type} - {project_id}")
                    if assessment['conditions']:
                        print(f"   Conditions: {', '.join(assessment['conditions'])}")
                else:
                    print(f"  Transmission project rejected: {project_type} - {project_id}")
                    print(f"   Reasons: {', '.join(assessment['reasons'])}")
    
    def _assess_transmission_project(self, project_id: str, project_type: str) -> dict:
        """
        Assess a transmission project for feasibility and compliance.
        
        Args:
            project_id: ID of the transmission project
            project_type: Type of transmission infrastructure
            
        Returns:
            Dictionary with approval status and conditions/reasons
        """
        # Simulate project assessment
        approval_probability = 0.7  # 70% approval rate
        
        if random.random() < approval_probability:
            conditions = []
            if random.random() < 0.3:
                conditions.append("Technical feasibility study required")
            if random.random() < 0.4:
                conditions.append("Community consultation needed")
            if random.random() < 0.2:
                conditions.append("Grid stability analysis required")
            if random.random() < 0.25:
                conditions.append("Cost-benefit analysis required")
            
            return {
                'approved': True,
                'conditions': conditions
            }
        else:
            reasons = [
                "Insufficient grid capacity",
                "Technical feasibility issues",
                "Community opposition",
                "Cost-benefit analysis unfavorable",
                "Regulatory compliance concerns",
                "Right-of-way acquisition challenges"
            ]
            return {
                'approved': False,
                'reasons': random.sample(reasons, random.randint(1, 2))
            }
    
    def _assess_grid_capacity(self) -> None:
        """Assess current grid capacity and identify bottlenecks."""
        print(f"Assessing grid capacity for year {self.timestep.year}")
        
        # Simulate capacity assessment
        total_capacity = random.uniform(8000, 12000)  # MW
        peak_demand = random.uniform(7000, 11000)  # MW
        reserve_margin = (total_capacity - peak_demand) / peak_demand
        
        print(f"  Total capacity: {total_capacity:.0f} MW")
        print(f"  Peak demand: {peak_demand:.0f} MW")
        print(f"  Reserve margin: {reserve_margin:.1%}")
        
        if reserve_margin < 0.15:  # Less than 15% reserve margin
            print("  Warning: Low reserve margin - capacity expansion needed")
            self._plan_capacity_expansion()
    
    def _plan_capacity_expansion(self) -> None:
        """Plan capacity expansion projects."""
        expansion_options = [
            "New transmission line construction",
            "Substation capacity upgrade",
            "Transformer replacement",
            "Grid interconnection project"
        ]
        
        selected_option = random.choice(expansion_options)
        estimated_cost = random.uniform(50, 200)  # Million dollars
        timeline = random.randint(2, 5)  # Years
        
        print(f"  Capacity expansion plan: {selected_option}")
        print(f"  Estimated cost: ${estimated_cost:.1f}M")
        print(f"  Timeline: {timeline} years")
    
    def _update_transmission_planning(self) -> None:
        """Update transmission planning based on current conditions."""
        print(f"Updating transmission planning for year {self.timestep.year}")
        
        # Simulate planning updates
        updates = []
        
        # Initialize target attributes if they don't exist
        if not hasattr(self.target, 'voltage_standards'):
            self.target.voltage_standards = {'transmission': 345}  # kV
        
        if not hasattr(self.target, 'reliability_standards'):
            self.target.reliability_standards = {'saifi': 1.2}  # System Average Interruption Frequency Index
        
        if random.random() < 0.4:  # 40% chance of voltage standard update
            old_voltage = self.target.voltage_standards['transmission']
            change = random.uniform(-5, 5)
            self.target.voltage_standards['transmission'] = max(115, min(765, old_voltage + change))
            updates.append(f"Transmission voltage: {old_voltage} kV -> {self.target.voltage_standards['transmission']:.0f} kV")
        
        if random.random() < 0.3:  # 30% chance of reliability standard update
            old_reliability = self.target.reliability_standards['saifi']
            change = random.uniform(-0.1, 0.1)
            self.target.reliability_standards['saifi'] = max(0.5, min(2.0, old_reliability + change))
            updates.append(f"SAIFI standard: {old_reliability:.2f} -> {self.target.reliability_standards['saifi']:.2f}")
        
        if updates:
            print(f"  Planning updates: {'; '.join(updates)}")
        else:
            print("  No planning changes this period")
    
    def _coordinate_regional_planning(self) -> None:
        """Coordinate with regional transmission organizations."""
        regions = ['Northeast', 'Southeast', 'Midwest', 'West', 'Texas']
        
        for region in regions:
            if random.random() < 0.3:  # 30% chance of regional coordination
                coordination_type = random.choice([
                    'interconnection_agreement',
                    'capacity_sharing',
                    'emergency_response',
                    'market_integration'
                ])
                print(f"Regional coordination: {coordination_type} with {region}")
    
    def _manage_maintenance_schedule(self) -> None:
        """Manage transmission line maintenance and upgrade schedules."""
        # Simulate maintenance activities
        num_maintenance_activities = random.randint(1, 4)
        
        for i in range(num_maintenance_activities):
            activity_type = random.choice([
                'line_inspection',
                'transformer_maintenance',
                'substation_upgrade',
                'vegetation_management',
                'corrosion_protection'
            ])
            location = f"transmission_facility_{i}"
            print(f"Maintenance: {activity_type} at {location}")
    
    def _assess_renewable_integration(self) -> None:
        """Assess renewable energy integration needs and challenges."""
        renewable_capacity = random.uniform(1000, 3000)  # MW
        total_capacity = random.uniform(8000, 12000)  # MW
        renewable_penetration = renewable_capacity / total_capacity
        
        print(f"Renewable integration assessment:")
        print(f"  Renewable capacity: {renewable_capacity:.0f} MW")
        print(f"  Renewable penetration: {renewable_penetration:.1%}")
        
        if renewable_penetration > 0.3:  # More than 30% renewable penetration
            print("  High renewable penetration - grid flexibility measures needed")
            self._plan_flexibility_measures()
    
    def _plan_flexibility_measures(self) -> None:
        """Plan grid flexibility measures for renewable integration."""
        flexibility_options = [
            "Battery energy storage system",
            "Demand response programs",
            "Smart grid technology upgrade",
            "Flexible transmission switching"
        ]
        
        selected_measures = random.sample(flexibility_options, random.randint(1, 2))
        for measure in selected_measures:
            cost = random.uniform(10, 100)  # Million dollars
            print(f"  Planned flexibility measure: {measure} (${cost:.1f}M)")

# Test the TransmissionPlanningEngine
print("Custom TransmissionPlanningEngine Example:")

# Create transmission planning authority and engine
tpa = TransmissionPlanningAgency(name="test_tpa")
engine = TransmissionPlanningEngine(tpa, project_review_probability=0.6)

# Mock timestep
engine.timestep = MockTimestep(2024)

# Run the engine
engine.run()

print(f"  Transmission corridors: {len(tpa.transmission_corridors)}")
print(f"  Grid investments: {len(tpa.grid_investments)}")
print("  TransmissionPlanningEngine created and tested successfully!")


## Part 4: Building a Complete Custom Simulation

Now let's put it all together and create a complete custom simulation that integrates our new agents and engines with the existing chance_c framework.


In [ ]:
# Custom Simulation Class
class CustomSimulation:
    """
    A custom simulation that extends the basic chance_c simulation with
    business agents and electricity transmission planning.
    """
    
    def __init__(self, config: SimulationConfig = None):
        """Initialize the custom simulation.
        
        Args:
            config: Simulation configuration (uses default if None)
        """
        self.config = config or SimulationConfig(
            simulation_name="Custom_ABM_Example",
            scenario="Business_Transmission",
            start_year=2020,
            n_years=5,
            agent_housing_aggregation=10
        )
        
        self.simulator = None
        self.business_agents = []
        self.tpa = None  # Transmission Planning Agency
    
    def setup_simulation(self) -> None:
        """Set up the custom simulation with all components."""
        print("Setting up custom simulation...")
        
        # Step 1: Create network first
        from pynsim import Network
        self.network = Network(name="custom_simulation_network")
        
        # Step 2: Create simulator with network
        self.simulator = ICOMSimulator(
            network=self.network,
            name=self.config.simulation_name,
            scenario=self.config.scenario,
            start_year=self.config.start_year,
            n_years=self.config.n_years
        )
        
        # Step 3: Set timestep information
        self.simulator.set_timestep_information()
        
        # Step 4: Set up landscape (simplified for demonstration)
        self._setup_mock_landscape()
        
        # Step 5: Create institutions
        self._create_institutions()
        
        # Step 6: Create agents
        self._create_agents()
        
        # Step 7: Add engines
        self._add_engines()
        
        print("Custom simulation setup complete!")
    
    def _setup_mock_landscape(self) -> None:
        """Set up a mock landscape for demonstration."""
        # In a real simulation, this would load actual geographic data
        # For demonstration, we'll create mock block groups
        print("Setting up mock landscape...")
        
        # Create mock block groups (simplified)
        self.network.mock_block_groups = [
            {'id': f'bg_{i}', 'population': random.randint(500, 2000)} 
            for i in range(10)
        ]
    
    def _create_institutions(self) -> None:
        """Create all institutions for the simulation."""
        print("Creating institutions...")
        
        # Standard institutions
        self.network.add_institution(AllHouseholdAgents(name='all_household_agents'))
        
        # Custom institutions
        self.tpa = TransmissionPlanningAgency(name='transmission_planning_agency')
        self.network.add_institution(self.tpa)
        
        # Business institution (to manage business agents)
        self.network.add_institution(Institution(name='all_business_agents'))
    
    def _create_agents(self) -> None:
        """Create all agents for the simulation."""
        print("Creating agents...")
        
        # Create household agents (simplified)
        for i in range(5):
            household = HouseholdAgent(
                name=f"household_{i}",
                location=f"bg_{i % 3}",  # Distribute across first 3 block groups
                income=random.uniform(30000, 100000)
            )
            self.network.add_component(household)
            self.network.get_institution('all_household_agents').add_component(household)
        
        # Create business agents
        business_types = ['retail', 'office', 'industrial', 'service']
        for i in range(8):
            business = BusinessAgent(
                name=f"business_{i}",
                location=f"bg_{i % 5}",  # Distribute across first 5 block groups
                business_type=random.choice(business_types),
                employees=random.randint(5, 50),
                revenue=random.uniform(100000, 2000000)
            )
            self.business_agents.append(business)
            self.network.add_component(business)
            self.network.get_institution('all_business_agents').add_component(business)
    
    def _add_engines(self) -> None:
        """Add all engines to the simulation."""
        print("Adding engines...")
        
        # Add standard engines (simplified versions)
        self.simulator.add_engine(BuildingDevelopment(self.network))
        
        # Add custom engines
        business_engine = BusinessLocationEngine(
            self.network,
            relocation_probability=0.2,
            sample_size=3
        )
        self.simulator.add_engine(business_engine)
        
        transmission_engine = TransmissionPlanningEngine(
            self.tpa,
            project_review_probability=0.4,
            planning_update_interval=3
        )
        self.simulator.add_engine(transmission_engine)
    
    def run_simulation(self) -> None:
        """Run the complete custom simulation."""
        print("\nStarting custom simulation run...")
        print("=" * 50)
        
        # Set up if not already done
        if self.simulator is None:
            self.setup_simulation()
        
        # Store business agents in the network for engines to access
        self.network.business_agents = self.business_agents
        
        # Run simulation
        try:
            self.simulator.start()
            print("\nCustom simulation completed successfully!")
            self._print_results()
        except Exception as e:
            print(f"\nSimulation failed: {e}")
            # In a real simulation, you'd want proper error handling
    
    def _print_results(self) -> None:
        """Print simulation results."""
        print("\nSimulation Results:")
        print("=" * 30)
        
        # Business statistics
        if hasattr(self.network, 'business_statistics'):
            stats = self.network.business_statistics
            print(f"Business statistics collected for {len(stats)} years")
            if stats:
                final_stats = stats[-1]
                print(f"Final year businesses: {final_stats['total_businesses']}")
                print(f"Business types: {final_stats['business_types']}")
        
        # Transmission statistics
        print(f"Transmission corridors designated: {len(self.tpa.transmission_corridors)}")
        print(f"Grid investments approved: {len(self.tpa.grid_investments)}")
        if hasattr(self.tpa, 'voltage_standards'):
            print(f"Current voltage standards: {self.tpa.voltage_standards}")
        if hasattr(self.tpa, 'reliability_standards'):
            print(f"Current reliability standards: {self.tpa.reliability_standards}")
        
        # Agent relocations
        relocations = 0
        for business in self.business_agents:
            if hasattr(business, 'relocation_history'):
                relocations += len(business.relocation_history)
        print(f"Business relocations: {relocations}")

# Test the complete custom simulation
print("Complete Custom Simulation Example:")
print("=" * 50)

# Create and run custom simulation
custom_sim = CustomSimulation()
custom_sim.setup_simulation()

# Show what was created
print(f"\nSimulation Components Created:")
print(f"  Institutions: {len(custom_sim.network.institutions)}")
print(f"  Household Agents: {len(custom_sim.network.get_institution('all_household_agents').components)}")
print(f"  Business Agents: {len(custom_sim.business_agents)}")
print(f"  Engines: {len(custom_sim.simulator.engines)}")

print(f"\nInstitutions:")
for inst_name in custom_sim.network.institutions:
    print(f"  - {inst_name}")

print(f"\nEngines:")
for engine in custom_sim.simulator.engines:
    print(f"  - {engine.__class__.__name__}")

print("\nReady to run custom simulation with custom_sim.run_simulation()!")


## Part 5: Integration Patterns and Best Practices

Here are key patterns and best practices for integrating custom components into chance_c simulations:


# Best Practices for Custom chance_c Components

## Agent Design
- Inherit from Component (urban agents) or Institution (institutional agents)
- Define _properties dict to track attributes over time
- Implement setup() method for timestep initialization
- Use descriptive names and clear documentation
- Keep decision-making logic modular and testable

## Engine Design
- Inherit from Engine and implement run() method
- Use self.timestep to access current simulation time
- Break complex logic into private helper methods
- Handle edge cases and validate inputs
- Log important events for debugging

## Institution Design
- Use institutions to group related agents
- Implement collective decision-making logic
- Manage shared resources and constraints
- Coordinate between different agent types
- Track institutional-level statistics

## Integration Patterns
- Store custom agents in appropriate institutions
- Use network properties to share data between engines
- Respect engine execution order dependencies
- Handle missing data gracefully
- Test components individually before integration

## Data Management
- Use network history to track time series data
- Store configuration in SimulationConfig objects
- Validate input data and parameters
- Export results in standard formats
- Document data requirements clearly

## Testing & Debugging
- Create unit tests for individual components
- Use mock objects for testing in isolation
- Add logging statements for key events
- Validate intermediate results
- Test with different parameter values

---

**Remember: Start simple and add complexity gradually!**


In [ ]:
# Example: Extending an Existing Engine
class EnhancedBuildingDevelopment(BuildingDevelopment):
    """
    An enhanced version of the BuildingDevelopment engine that includes
    transmission impact assessment and business zoning considerations.
    """
    
    def __init__(self, target, tpa_institution=None, **kwargs):
        """Initialize enhanced building development engine.
        
        Args:
            target: The simulation network target
            tpa_institution: Transmission planning agency for load assessments
        """
        super().__init__(target, **kwargs)
        self.tpa = tpa_institution
        self.development_proposals = []
    
    def run(self) -> None:
        """Execute enhanced building development with transmission load checks."""
        logging.info(f"Running Enhanced Building Development, year {self.timestep.year}")
        
        # Step 1: Identify development needs (from parent class)
        development_needed = self._identify_development_needs()
        
        # Step 2: Assess transmission load impact for each development
        approved_developments = []
        for development in development_needed:
            if self._assess_transmission_impact(development):
                approved_developments.append(development)
        
        # Step 3: Execute approved developments
        for development in approved_developments:
            self._execute_development(development)
        
        # Step 4: Update development statistics
        self._update_development_statistics(approved_developments)
    
    def _identify_development_needs(self) -> List[Dict]:
        """Identify where development is needed."""
        developments = []
        
        # Check each block group for development needs (simplified)
        for i in range(5):  # Mock block groups
            block_group_id = f"bg_{i}"
            
            # Simulate demand exceeding supply
            if random.random() < 0.3:  # 30% chance of needing development
                development = {
                    'location': block_group_id,
                    'type': random.choice(['residential', 'commercial', 'mixed_use']),
                    'units': random.randint(10, 50)
                }
                developments.append(development)
        
        return developments
    
    def _assess_transmission_impact(self, development: Dict) -> bool:
        """Assess transmission load impact of proposed development."""
        if self.tpa is None:
            return True  # No transmission review
        
        # Estimate load growth from development
        load_growth = development['units'] * random.uniform(2, 8)  # MW per unit
        
        assessment = self.tpa.assess_transmission_needs(
            development['location'], 
            load_growth
        )
        
        if assessment['approved']:
            print(f"Development approved: {development['type']} at {development['location']}")
            return True
        else:
            print(f"Development rejected: {development['type']} at {development['location']}")
            print(f"   Reasons: {', '.join(assessment['reasons'])}")
            return False
    
    def _execute_development(self, development: Dict) -> None:
        """Execute approved development."""
        print(f"Building {development['units']} {development['type']} units at {development['location']}")
        
        # Record development in history
        if not hasattr(self.target, 'development_history'):
            self.target.development_history = []
        
        development['year'] = self.timestep.year
        self.target.development_history.append(development)
    
    def _update_development_statistics(self, developments: List[Dict]) -> None:
        """Update development statistics."""
        total_units = sum(dev['units'] for dev in developments)
        
        print(f"Year {self.timestep.year}: {len(developments)} developments, {total_units} total units")

# Example: Creating a Custom Configuration
class CustomSimulationConfig(SimulationConfig):
    """Extended configuration class with custom parameters."""
    
    def __init__(self, **kwargs):
        # Separate custom parameters from standard SimulationConfig parameters
        custom_params = [
            'business_relocation_rate', 'transmission_project_review_rate', 
            'transmission_planning_update_interval', 'max_businesses_per_block_group',
            'business_startup_rate', 'capacity_threshold', 'voltage_standards', 
            'reliability_standards'
        ]
        
        # Extract custom parameters
        custom_kwargs = {}
        standard_kwargs = {}
        
        for key, value in kwargs.items():
            if key in custom_params:
                custom_kwargs[key] = value
            else:
                standard_kwargs[key] = value
        
        # Initialize base configuration with only standard parameters
        super().__init__(**standard_kwargs)
        
        # Add custom parameters
        self.business_relocation_rate = custom_kwargs.get('business_relocation_rate', 0.15)
        self.transmission_project_review_rate = custom_kwargs.get('transmission_project_review_rate', 0.4)
        self.transmission_planning_update_interval = custom_kwargs.get('transmission_planning_update_interval', 3)
        
        # Business development parameters
        self.max_businesses_per_block_group = custom_kwargs.get('max_businesses_per_block_group', 20)
        self.business_startup_rate = custom_kwargs.get('business_startup_rate', 0.05)
        
        # Transmission planning parameters
        self.capacity_threshold = custom_kwargs.get('capacity_threshold', 0.85)
        self.voltage_standards = custom_kwargs.get('voltage_standards', {'transmission': 345})
        self.reliability_standards = custom_kwargs.get('reliability_standards', {'saifi': 1.2})

# Demonstrate the enhanced components
print("Enhanced Components Example:")
print("=" * 40)

# Create enhanced configuration
enhanced_config = CustomSimulationConfig(
    simulation_name="Enhanced_Custom_Simulation",
    business_relocation_rate=0.2,
    transmission_project_review_rate=0.5,
    capacity_threshold=0.8
)

print(f"Enhanced Configuration:")
print(f"  Business relocation rate: {enhanced_config.business_relocation_rate}")
print(f"  Transmission project review rate: {enhanced_config.transmission_project_review_rate}")
print(f"  Capacity threshold: {enhanced_config.capacity_threshold}")

# Create enhanced engine
tpa = TransmissionPlanningAgency(name="enhanced_tpa")
enhanced_engine = EnhancedBuildingDevelopment(target=MockTarget(), tpa_institution=tpa)
enhanced_engine.timestep = MockTimestep(2024)

# Test enhanced engine
enhanced_engine.run()

print("Enhanced components created successfully!")


## Summary and Next Steps

Congratulations! You've learned how to design custom simulations in chance_c. Here's what we covered:

### Key Concepts Learned

1. **Architecture Understanding**: How pynsim components work together in chance_c
2. **Custom Agents**: Creating both urban agents (BusinessAgent) and institutional agents (EPA)
3. **Custom Engines**: Building behavioral modules that execute simulation logic
4. **Integration Patterns**: Connecting custom components with existing chance_c infrastructure
5. **Advanced Techniques**: Extending the Model class for seamless integration

### What You Can Do Now

**Create Custom Urban Agents** - Design agents with specific decision-making logic  
**Build Institutional Agents** - Manage collective resources and regulations  
**Develop Custom Engines** - Implement specialized behavioral models  
**Integrate with chance_c** - Add your components to existing simulations  
**Extend Configurations** - Create custom parameters and settings  

### Next Steps for Your Research

1. **Start Simple**: Begin with one custom agent or engine
2. **Test Thoroughly**: Use unit tests and mock objects for validation
3. **Document Everything**: Clear documentation helps future development
4. **Iterate Gradually**: Add complexity incrementally
5. **Share Your Work**: Contribute back to the chance_c community

### Resources for Further Development

- **chance_c Documentation**: Core framework reference
- **pynsim Documentation**: Underlying simulation engine
- **Example Scripts**: Study existing chance_c implementations
- **Community Support**: Engage with other chance_c developers


# Quick Reference: Key Classes and Methods

## Quick Reference for Custom chance_c Development

### Base Classes

| Class | Description |
|-------|-------------|
| `Component` | Base class for urban agents (from pynsim) |
| `Institution` | Base class for institutional agents (from pynsim) |
| `Engine` | Base class for behavioral engines (from pynsim) |
| `Model` | Main simulation class (from chance_c) |
| `SimulationConfig` | Configuration management (from chance_c) |

### Key Methods to Implement

| Method | Description |
|--------|-------------|
| `__init__()` | Initialize your custom component with parameters |
| `setup(timestep)` | Set up component for each timestep |
| `run()` | Main execution method for engines |
| `_properties` | Dict defining trackable attributes over time |

### Network Access Patterns

| Pattern | Description |
|---------|-------------|
| `self.target.nodes` | Access block group nodes |
| `self.target.components` | Access all agents/components |
| `self.target.get_institution(name)` | Get specific institution |
| `self.target.add_component(agent)` | Add agent to network |
| `self.target.add_institution(inst)` | Add institution to network |

### Data Management

| Method | Description |
|--------|-------------|
| `self.timestep.year` | Current simulation year |
| `network.get_history(name)` | Get time series data |
| `component.get_history(name)` | Get component history |
| `network._properties` | View available histories |

---

**Remember:**
- Always inherit from the appropriate base class!
- Use `logging.info()` to track important events during simulation
- Test your components individually before integrating

**You're now ready to create your own custom chance_c simulations!**
